# 不计分实验 - 正则化逻辑回归的梯度

## 目标
在本实验中，你将：
- 扩展梯度计算的实现，使其包含正则化。
- 编写一个使用循环的版本
- 选择性编写一个向量化版本
- 对二者的速度差异进行测试

In [1]:
import numpy as np
from lab_utils import sigmoid

## 循环版本

正则化代价函数的梯度，是代价相对于参数 $w$ 和 $b$ 的偏导数：

$$\frac{\partial J(\mathbf{w})}{\partial b} = \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})  \tag {1}$$

$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \left( \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) x_j^{(i)} \right) + \lambda w_j  \quad\, \mbox{for $j=0...(n-1)$} \tag {2}$$


你将实现一个名为 `compute_gradient_reg` 的函数，它将返回 $\frac{\partial J(\mathbf{w},b)}{\partial w},\frac{\partial J(\mathbf{w},b)}{\partial b}$。

请完成 `compute_gradient_reg` 函数：

- 按照先前实验中 `compute_gradient` 函数的做法，计算每个元素 `dJdw` 和 `dJdb` 的梯度：
    - 初始化用于累加 `dJdw` 和 `dJdb` 的变量
    - 遍历所有样本
        - 计算该样本的误差 $g(\mathbf{x}^{(i)T}\mathbf{w} + b) - \mathbf{y}^{(i)}$
        - 将误差加到 `dJdb`（上面的公式 1）
        - 对该样本中的每个输入值 $x_{j}^{(i)}$，  
            - 将误差乘以输入 $x_{j}^{(i)}$，并加到 `dJdw` 的对应元素中。（上面的公式 2）
     - 将 `dJdb` 和 `dJdw` 除以样本总数（m）
- 现在计算正则化项
    - 遍历所有 $w$
        - 将 $\lambda * w_j$ 加到 `dJdw` 的对应元素中

进行这些操作时，请记住变量 X 和 y 不是标量，而是形状分别为（$m, n$）和（$𝑚$,1）的矩阵，其中 $𝑛$ 是特征数量，$𝑚$ 是训练样本数量。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>
    
```python     
def compute_gradient_reg(X, y, w, b, lambda_ = 1): 
    """
    Computes the gradient for linear regression 
 
    Args:
      X : (array_like Shape (m,n)) variable such as house size 
      y : (array_like Shape (m,1)) actual value 
      w : (array_like Shape (n,1)) values of parameters of the model      
      b : (scalar)                 value of parameter of the model  
      lambda_ : (scalar,float)      regularization constant
    Returns
      dJdw: (array_like Shape (n,1)) The gradient of the cost w.r.t. the parameters w. 
      dJdb: (scalar)                The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape
    dJdw = np.zeros((n,1))
    dJdb = 0.
    err  = 0.

    ### START CODE HERE ### 
    for i in range(m):
        err = sigmoid(X[i] @ w + b)  - y[i]    
        for j in range(n):
            dJdw[j] = dJdw[j] + err * X[i][j]
        dJdb = dJdb + err
    dJdw = dJdw/m
    dJdb = dJdb/m
    
    for j in range(n):
        dJdw[j] = dJdw[j] + lambda_ * w[j]

    ### END CODE HERE ###         
        
    return dJdb[0],dJdw  #index dJdb to return scalar value
```
</details>

In [2]:
def compute_gradient_reg(X, y, w, b, lambda_ = 1): 
    """
    Computes the gradient for linear regression 
 
    Args:
      X : (array_like Shape (m,n)) variable such as house size 
      y : (array_like Shape (m,1)) actual value 
      w : (array_like Shape (n,1)) values of parameters of the model      
      b : (scalar)                 value of parameter of the model  
      lambda_ : (scalar,float)      regularization constant
    Returns
      dJdw: (array_like Shape (n,1)) The gradient of the cost w.r.t. the parameters w. 
      dJdb: (scalar)                The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape
    dJdw = np.zeros((n,1))
    dJdb = 0.
    err  = 0.

    ### START CODE HERE ### 

    ### END CODE HERE ###         
        
    return dJdb[0],dJdw  #index dJdb to return scalar value

运行下面的单元格，检查 `compute_gradient_reg` 函数的实现。

In [3]:
np.random.seed(1)
X_tmp = np.random.rand(5,3)
y_tmp = np.array([0,1,0,1,0]).reshape(-1,1)
initial_w  = np.random.rand(X_tmp.shape[1]).reshape(-1,1)
initial_b = 0.5
lambda_ = 1
dJdb, dJdw =  compute_gradient_reg(X_tmp, y_tmp, initial_w, initial_b, lambda_)

print(f"dJdb: {dJdb}", )
print(f"Regularized dJdw:\n {dJdw.tolist()}", )

dJdb: 0.341798994972791
Regularized dJdw:
 [[0.7504021880933689], [0.6789572088513987], [0.5882363864318614]]


**预期输出**
```
dJdb: 0.341798994972791
Regularized dJdw:
 [[0.7504021880933689], [0.6789572088513987], [0.5882363864318614]]
 ```

## 向量化版本

利用上面的公式以及在 Lab04 中使用向量化梯度下降的经验，编写 `compute_gradient_reg_matrix`。

第一部分与实验 4 相同。添加最后一部分，将 $\lambda * w_j$ 加到 $djdw$ 的对应元素中。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>
    
```python     
def compute_gradient_reg_matrix(X, y, w, b, lambda_= 1): 
    """
    Computes the gradient for linear regression 
 
    Args:
      X : (array_like Shape (m,n)) variable such as house size 
      y : (array_like Shape (m,1)) actual value 
      w : (array_like Shape (n,1)) Values of parameters of the model      
      b : (scalar )                Values of parameter of the model      
      predict_function: (function) function to call to make prediction
    Returns
      dJdw: (array_like Shape (n,1)) The gradient of the cost w.r.t. the parameters w. 
      dJdb: (scalar)                The gradient of the cost w.r.t. the parameter b. 
                                  
    """
    m,n = X.shape
    ### START CODE HERE ### 
    ### BEGIN SOLUTION ###
    f_wb = sigmoid(X @ w + b) 
    e   = f_wb - y                 ##None
    dJdw  = (1/m) * (X.T @ e)      ##None
    dJdb  = (1/m) * np.sum(e)      ##None
    for j in range(n):
        dJdw[j] = dJdw[j] + lambda_ * w[j]
    ### END SOLUTION ### 
    ### END CODE HERE ###         
   
    return dJdb,dJdw
```
</details>

In [4]:
def compute_gradient_reg_matrix(X, y, w, b, lambda_= 1): 
    """
    Computes the gradient for linear regression 
 
    Args:
      X : (array_like Shape (m,n)) variable such as house size 
      y : (array_like Shape (m,1)) actual value 
      w : (array_like Shape (n,1)) Values of parameters of the model      
      b : (scalar )                Values of parameter of the model      
      predict_function: (function) function to call to make prediction
    Returns
      dJdw: (array_like Shape (n,1)) The gradient of the cost w.r.t. the parameters w. 
      dJdb: (scalar)                The gradient of the cost w.r.t. the parameter b. 
                                  
    """
    m,n = X.shape
    ### START CODE HERE ### 

    ### END CODE HERE ###         
   
    return dJdb,dJdw

In [6]:
np.random.seed(1)
X_tmp = np.random.rand(5,3)
y_tmp = np.array([0,1,0,1,0]).reshape(-1,1)
initial_w  = np.random.rand(X_tmp.shape[1]).reshape(-1,1)
initial_b = 0.5
lambda_ = 1
dJdb, dJdw =  compute_gradient_reg_matrix(X_tmp, y_tmp, initial_w, initial_b, lambda_)

print(f"dJdb: {dJdb}", )
print(f"Regularized dJdw:\n {dJdw.tolist()}", )

dJdb: 0.34179899497279104
Regularized dJdw:
 [[0.7504021880933689], [0.6789572088513987], [0.5882363864318614]]


**预期输出**
```
dJdb: 0.34179899497279104
Regularized dJdw:
 [[0.7504021880933689], [0.6789572088513987], [0.5882363864318614]]
 ```

## 对您的实现进行速度测试

In [7]:
import time

np.random.seed(1)
m = 500; n=100
X_tmp = np.random.rand(m,n)
y_tmp = np.zeros((m,1))
initial_w  = np.random.rand(X_tmp.shape[1]).reshape(-1,1)
initial_b = 0.5
lambda_ = 1

start = time.time()
for i in range(10):
    dJdb, dJdw =  compute_gradient_reg(X_tmp, y_tmp, initial_w, initial_b, lambda_)
end = time.time()
print(f"non- matrix time {end - start}")


start = time.time()
for i in range(10):
    dJdb, dJdw =  compute_gradient_reg_matrix(X_tmp, y_tmp, initial_w, initial_b, lambda_)
end = time.time()
print(f"matrix time      {end - start}")


non- matrix time 7.255051374435425
matrix time      0.2189960479736328


# 